In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as transforms

from PIL import Image
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)

TARGETS       = ["Dry_Green_g", "Dry_Dead_g", "Dry_Clover_g", "GDM_g", "Dry_Total_g"]
WEIGHT_VECTOR = np.array([0.1, 0.1, 0.1, 0.2, 0.5], dtype=np.float32)

DATASET_DIR   = "/kaggle/input/competitions/csiro-biomass"
TRAIN_CSV     = os.path.join(DATASET_DIR, "train.csv")
TEST_CSV      = os.path.join(DATASET_DIR, "test.csv")
TRAIN_IMG_DIR = os.path.join(DATASET_DIR, "train")
TEST_IMG_DIR  = os.path.join(DATASET_DIR, "test")
WEIGHTS_PATH  = "/kaggle/input/datasets/mayaevensen/resnet50-pretrained-weights/resnet50-11ad3fa6.pth"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

train.csv   : True
test.csv    : True
train dir   : True
test  dir   : True


In [2]:
class ResNet50Extractor:
    """Mirrors works_with_kaggle.ipynb: pretrained if available, else random init.
       Concatenates features from left/right halves of the 2000x1000 image -> 4096-d."""
    def __init__(self, device="cpu"):
        self.device = device
        try:
            weights = models.ResNet50_Weights.DEFAULT
            resnet = models.resnet50(weights=weights)
            print("Loaded pretrained ResNet50 weights.")
        except Exception as e:
            print("No internet -> random init fallback:", e)
            resnet = models.resnet50(weights=None)

        self.model = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        self.preprocess = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225]),
        ])

    @torch.no_grad()
    def _one(self, pil_img):
        t = self.preprocess(pil_img).unsqueeze(0).to(self.device)
        return self.model(t).flatten().cpu().numpy().astype(np.float32)

    def get_features(self, image_path):
        if not os.path.exists(image_path):
            print("Missing:", image_path)
            return np.zeros(4096, dtype=np.float32)
        try:
            img = Image.open(image_path).convert("RGB")
            left  = img.crop((0, 0, 1000, 1000))
            right = img.crop((1000, 0, 2000, 1000))
            return np.concatenate([self._one(left), self._one(right)], axis=0).astype(np.float32)
        except Exception as e:
            print(f"Error {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

train_df, y_all = load_train_data(TRAIN_CSV)
print("Train rows:", len(train_df), " y:", y_all.shape)

extractor = ResNet50Extractor(device=device)

all_feats, all_ids = [], []
for i, row in enumerate(train_df.itertuples(index=False), 1):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row.image_id}.jpg")
    all_feats.append(extractor.get_features(img_path))
    all_ids.append(row.image_id)
    if i % 100 == 0:
        print(f"  {i}/{len(train_df)}")

X_all = np.asarray(all_feats, dtype=np.float32)
print("X_all:", X_all.shape, " y_all:", y_all.shape)

np.save("/kaggle/working/X_all.npy", X_all)
np.save("/kaggle/working/y_all.npy", y_all)

Device: cpu
Train rows: 357  y: (357, 5)
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
No internet -> random init fallback: <urlopen error [Errno -3] Temporary failure in name resolution>
  100/357
  200/357
  300/357
X_all: (357, 4096)  y_all: (357, 5)


In [ ]:
def load_train_data(csv_path):
    df = pd.read_csv(csv_path)
    df["image_id"] = df["sample_id"].str.split("__").str[0]
    df_wide = df.pivot_table(
        index=["image_id", "image_path"],
        columns="target_name",
        values="target",
    ).reset_index()
    return df_wide, df_wide[TARGETS].values.astype(np.float32)


def weighted_global_r2(y_true, y_pred):
    w  = WEIGHT_VECTOR.astype(np.float64)
    yt, yp = y_true.reshape(-1), y_pred.reshape(-1)
    ww     = np.repeat(w, y_true.shape[0])
    ybar   = np.sum(ww * yt) / np.sum(ww)
    ss_res = np.sum(ww * (yt - yp) ** 2)
    ss_tot = np.sum(ww * (yt - ybar) ** 2) + 1e-12
    return float(1.0 - ss_res / ss_tot)


def rmse_per_target(y_true, y_pred):
    return {t: float(np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i])))
            for i, t in enumerate(TARGETS)}


def prepare_submission(test_csv_path, predictions, image_ids):
    df_test = pd.read_csv(test_csv_path)
    pred_dict = {img_id: dict(zip(TARGETS, row))
                 for img_id, row in zip(image_ids, predictions)}
    def get_pred(row):
        img_id = row["sample_id"].split("__")[0]
        return max(0.0, float(pred_dict.get(img_id, {}).get(row["target_name"], 0.0)))
    df_test["target"] = df_test.apply(get_pred, axis=1)
    return df_test[["sample_id", "target"]]

In [ ]:
class ResNet50Extractor:
    """
    Loads pretrained ResNet50 from local .pth file (no internet needed).
    Splits each 2000x1000 image into left/right halves -> 2x2048 = 4096-d vector.
    Supports horizontal flip for augmentation.
    """
    PREPROCESS = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225]),
    ])

    def __init__(self, weights_path, device="cpu"):
        self.device = device
        resnet = models.resnet50(weights=None)
        resnet.load_state_dict(torch.load(weights_path, map_location="cpu"))
        self.backbone = nn.Sequential(*list(resnet.children())[:-1]).to(device).eval()
        print("Loaded pretrained ResNet50 weights.")

    @torch.no_grad()
    def _extract(self, crop):
        t = self.PREPROCESS(crop).unsqueeze(0).to(self.device)
        return self.backbone(t).flatten().cpu().numpy().astype(np.float32)

    def get_features(self, image_path, hflip=False):
        if not os.path.exists(image_path):
            return np.zeros(4096, dtype=np.float32)
        try:
            img = Image.open(image_path).convert("RGB")
            if hflip:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            left  = img.crop((0,    0, 1000, 1000))
            right = img.crop((1000, 0, 2000, 1000))
            return np.concatenate([self._extract(left), self._extract(right)])
        except Exception as e:
            print(f"Error loading {image_path}: {e}")
            return np.zeros(4096, dtype=np.float32)

In [ ]:
train_df, y_orig = load_train_data(TRAIN_CSV)
n = len(train_df)
print(f"Training images: {n}")

extractor = ResNet50Extractor(WEIGHTS_PATH, device=DEVICE)

# Extract original + horizontally flipped for each train image.
# Validation always uses originals only — flips are train-only augmentation.
X_orig = np.zeros((n, 4096), dtype=np.float32)
X_flip = np.zeros((n, 4096), dtype=np.float32)

for i, row in enumerate(train_df.itertuples(index=False)):
    img_path = os.path.join(TRAIN_IMG_DIR, f"{row.image_id}.jpg")
    X_orig[i] = extractor.get_features(img_path, hflip=False)
    X_flip[i] = extractor.get_features(img_path, hflip=True)
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/{n}")

print("Train features done:", X_orig.shape)

# Test features
df_test  = pd.read_csv(TEST_CSV)
df_test["image_id"] = df_test["sample_id"].str.split("__").str[0]
test_unique = df_test.drop_duplicates("image_id").copy()

X_test, test_ids = [], []
for row in test_unique.itertuples(index=False):
    img_path = os.path.join(TEST_IMG_DIR, f"{row.image_id}.jpg")
    X_test.append(extractor.get_features(img_path, hflip=False))
    test_ids.append(row.image_id)
X_test = np.array(X_test, dtype=np.float32)
print("Test features done:", X_test.shape)

Train: (285, 4096)  Val: (72, 4096)
  ep   1  tr=5.4829  val_R2=-0.0996
  ep  10  tr=3.6304  val_R2=0.2965
  ep  20  tr=3.4393  val_R2=0.3237
  ep  30  tr=3.5439  val_R2=0.2919
  ep  40  tr=3.4961  val_R2=0.3261
  ep  50  tr=3.3810  val_R2=0.3266
  ep  60  tr=3.3785  val_R2=0.3231
  ep  70  tr=3.3748  val_R2=0.3279
  ep  80  tr=3.3762  val_R2=0.3282
Validation weighted R2: 0.3281960797913678
Validation RMSE/target: {'Dry_Green_g': 20.969039888189105, 'Dry_Dead_g': 12.809950374559783, 'Dry_Clover_g': 14.899184770351413, 'GDM_g': 21.634316397793675, 'Dry_Total_g': 23.506142857568896}


In [ ]:
class BiomassNet(nn.Module):
    """
    MLP with BatchNorm + GELU + Dropout in each hidden block.
    Deliberately not too wide given the small dataset (~357 samples).
    """
    def __init__(self, input_dim=4096, hidden_dims=(512, 256, 128),
                 output_dim=5, dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.GELU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

  ep   1  tr=4.7088
  ep  10  tr=3.3834
  ep  20  tr=3.3191
  ep  30  tr=3.3141
  ep  40  tr=3.3242
  ep  50  tr=3.2987
  ep  60  tr=3.3234
  ep  70  tr=3.2665
  ep  80  tr=3.3092
Unique test images: 1
                    sample_id     target
0  ID1001187975__Dry_Clover_g   2.499922
1    ID1001187975__Dry_Dead_g   8.623918
2   ID1001187975__Dry_Green_g  18.718363
3   ID1001187975__Dry_Total_g  37.560284
4         ID1001187975__GDM_g  24.971817
Saved /kaggle/working/submission.csv  rows: 5
Working dir: ['X_all.npy', 'submission.csv', 'y_all.npy', '__notebook__.ipynb']


In [ ]:
_LOSS_W = torch.tensor(WEIGHT_VECTOR, dtype=torch.float32)

def weighted_smooth_l1(pred, target, beta=1.0):
    loss = F.smooth_l1_loss(pred, target, beta=beta, reduction="none")
    return (loss * _LOSS_W.to(pred.device)).mean()


def train_fold(X_tr, y_tr, X_val, y_val, feat_scaler, tgt_scaler,
               epochs=200, batch_size=32, lr=3e-4, weight_decay=1e-3,
               patience=30, device="cpu"):
    """
    Trains one BiomassNet. Scalers are pre-fit; this function only transforms.
    Returns (model, best_val_r2, oof_preds_in_original_scale).
    """
    X_tr_s  = feat_scaler.transform(X_tr).astype(np.float32)
    X_val_s = feat_scaler.transform(X_val).astype(np.float32)
    y_tr_s  = tgt_scaler.transform(y_tr).astype(np.float32)

    def loader(X, y, shuffle):
        ds = TensorDataset(torch.tensor(X), torch.tensor(y))
        return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

    tr_loader  = loader(X_tr_s, y_tr_s, shuffle=True)
    val_loader = loader(X_val_s, y_val.astype(np.float32), shuffle=False)

    model = BiomassNet(input_dim=X_tr.shape[1]).to(device)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=lr/100)

    best_r2, best_state, wait = -np.inf, None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        for xb, yb in tr_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            weighted_smooth_l1(model(xb), yb).backward()
            opt.step()
        sched.step()

        model.eval()
        preds = []
        with torch.no_grad():
            for xb, _ in val_loader:
                preds.append(model(xb.to(device)).cpu().numpy())
        preds_orig = tgt_scaler.inverse_transform(np.concatenate(preds))
        r2 = weighted_global_r2(y_val, preds_orig)

        if r2 > best_r2:
            best_r2   = r2
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                print(f"    Early stop at epoch {epoch}  best R2={best_r2:.4f}")
                break

        if epoch == 1 or epoch % 25 == 0:
            print(f"    ep {epoch:3d}  val_R2={r2:.4f}  best={best_r2:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        final_preds_s = model(torch.tensor(X_val_s).to(device)).cpu().numpy()
    return model, best_r2, tgt_scaler.inverse_transform(final_preds_s)

In [ ]:
N_FOLDS    = 5
kf         = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
oof_preds  = np.zeros_like(y_orig)
test_preds = np.zeros((len(test_unique), 5))
fold_scores, fold_scalers = [], []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X_orig), 1):
    print(f"\n{'='*55}\n  FOLD {fold}/{N_FOLDS}\n{'='*55}")

    # Train uses original + flipped; val uses original only
    X_tr = np.concatenate([X_orig[tr_idx], X_flip[tr_idx]])
    y_tr = np.concatenate([y_orig[tr_idx], y_orig[tr_idx]])
    X_val, y_val = X_orig[val_idx], y_orig[val_idx]

    # Fit scalers on training data only (no val leakage)
    feat_scaler = StandardScaler().fit(X_tr)
    tgt_scaler  = StandardScaler().fit(y_tr)
    fold_scalers.append((feat_scaler, tgt_scaler))

    model, best_r2, val_preds = train_fold(
        X_tr, y_tr, X_val, y_val,
        feat_scaler=feat_scaler, tgt_scaler=tgt_scaler,
        epochs=200, batch_size=32, lr=3e-4, weight_decay=1e-3,
        patience=30, device=DEVICE,
    )

    oof_preds[val_idx] = val_preds
    fold_scores.append(best_r2)
    print(f"  Fold {fold} best R2: {best_r2:.4f}")

    # Accumulate test predictions (averaged across folds)
    X_test_s = feat_scaler.transform(X_test).astype(np.float32)
    with torch.no_grad():
        tp_s = model(torch.tensor(X_test_s).to(DEVICE)).cpu().numpy()
    test_preds += tgt_scaler.inverse_transform(tp_s) / N_FOLDS

print(f"\n{'='*55}")
print(f"OOF weighted R2 : {weighted_global_r2(y_orig, oof_preds):.4f}")
print(f"Per-fold R2     : {[round(s, 4) for s in fold_scores]}")
print(f"RMSE per target : {rmse_per_target(y_orig, oof_preds)}")

In [ ]:
test_preds_clipped = np.clip(test_preds, 0, None)  # biomass can't be negative

submission = prepare_submission(TEST_CSV, test_preds_clipped, test_ids)
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv — rows:", len(submission))
print(submission.head(10))